Now we need the second classifier to do a full webscraper. Then confirm the relevance to mining activities in Zambia. Then try and classify the impacts using Mehrnoosh's framework. 

In [2]:
import requests
import trafilatura
import json
import re
from bs4 import BeautifulSoup
from dateutil import parser as dtparser
from urllib.parse import urlparse

# Try to import Newspaper3k as an optional fallback extractor
try:
    from newspaper import Article as _NPArticle
    _HAS_NEWSPAPER = True
except Exception:
    _HAS_NEWSPAPER = False


# ------------------ DATE EXTRACTION ------------------ #

_MONTHS = (
    "January|February|March|April|May|June|July|August|September|October|November|December"
)

def fallback_date_from_url(url: str) -> str:
    """
    Best-effort fallback date extraction from URL patterns.
    Returns ISO date 'YYYY-MM-DD' or 'unknown'.

    Supports:
      - /YYYY/MM/DD/  (common on WordPress, e.g., colombogazette.com/2025/12/08/...)
      - /afp/YYMMDD... (SpaceWar AFP wire URLs, e.g., /afp/251208073123....html => 2025-12-08)
    """
    try:
        path = urlparse(url).path or ""
    except Exception:
        return "unknown"

    # Pattern 1: /YYYY/MM/DD/
    m = re.search(r"/(20\d{2})/(0[1-9]|1[0-2])/(0[1-9]|[12]\d|3[01])/", path)
    if m:
        yyyy, mm, dd = m.group(1), m.group(2), m.group(3)
        return f"{yyyy}-{mm}-{dd}"

    # Pattern 2: /afp/YYMMDD...
    m = re.search(r"/afp/(\d{2})(\d{2})(\d{2})", path)
    if m:
        yy, mm, dd = m.group(1), m.group(2), m.group(3)
        try:
            yyyy = 2000 + int(yy)
            return f"{yyyy:04d}-{mm}-{dd}"
        except Exception:
            return "unknown"

    return "unknown"


def extract_published_date_from_html(html: str) -> str:
    """
    Best-effort extraction of published date from HTML metadata.
    Returns ISO date 'YYYY-MM-DD' or 'unknown'.

    Strategy:
      1) JSON-LD (datePublished etc., then dateModified fallback)
      2) meta tags (expanded list inc. DC/dcterms)
      3) <time datetime="...">
      4) conservative visible-text fallback near top of page
    """
    def _to_iso(s: str) -> str | None:
        try:
            dt = dtparser.parse(s, fuzzy=True)
            return dt.date().isoformat()
        except Exception:
            return None

    soup = BeautifulSoup(html, "html.parser")

    # 1) JSON-LD datePublished (and friends)
    # Some sites embed multiple objects and/or @graph
    jsonld_keys_primary = ("datePublished", "dateCreated", "uploadDate")
    jsonld_keys_fallback = ("dateModified",)

    for script in soup.find_all("script", type=re.compile("ld\\+json", re.I)):
        if not script.string:
            continue
        try:
            data = json.loads(script.string)
        except Exception:
            continue

        objs = data if isinstance(data, list) else [data]
        # Expand @graph objects if present
        expanded: list[dict] = []
        for obj in objs:
            if isinstance(obj, dict):
                expanded.append(obj)
                graph = obj.get("@graph")
                if isinstance(graph, list):
                    expanded.extend([g for g in graph if isinstance(g, dict)])
        # Try primary keys first
        for obj in expanded:
            for key in jsonld_keys_primary:
                if isinstance(obj.get(key), str):
                    iso = _to_iso(obj[key])
                    if iso:
                        return iso
        # Then fallback keys (dateModified etc.)
        for obj in expanded:
            for key in jsonld_keys_fallback:
                if isinstance(obj.get(key), str):
                    iso = _to_iso(obj[key])
                    if iso:
                        return iso

    # 2) Common meta tags (expanded)
    meta_candidates = [
        # OpenGraph / article
        ("property", "article:published_time"),
        ("property", "og:published_time"),
        ("property", "og:article:published_time"),
        ("property", "article:modified_time"),   # fallback if published missing
        ("property", "og:updated_time"),         # fallback

        # Parsely / generic
        ("name", "parsely-pub-date"),
        ("name", "pubdate"),
        ("name", "publishdate"),
        ("name", "publish-date"),
        ("name", "date"),
        ("itemprop", "datePublished"),
        ("itemprop", "dateCreated"),

        # Dublin Core / dcterms (very common on wire/aggregators)
        ("name", "dc.date"),
        ("name", "dc.date.issued"),
        ("name", "dc.date.created"),
        ("name", "dcterms.issued"),
        ("name", "dcterms.created"),
    ]

    for attr, key in meta_candidates:
        tag = soup.find("meta", attrs={attr: key})
        if tag and tag.get("content"):
            iso = _to_iso(tag["content"])
            if iso:
                return iso

    # 3) <time datetime="..."> (often works)
    # Try multiple <time> tags, not just the first
    for t in soup.find_all("time"):
        if t.get("datetime"):
            iso = _to_iso(t["datetime"])
            if iso:
                return iso
        txt = t.get_text(strip=True)
        if txt:
            iso = _to_iso(txt)
            if iso:
                return iso

    # 4) Conservative visible-text fallback near top of page
    # Only scan the first chunk of visible text to avoid grabbing random dates in body.
    # Accept common long-form date patterns.
    top_text = soup.get_text(" ", strip=True)
    top_text = top_text[:1800]  # keep it conservative

    patterns = [
        rf"\b\d{{1,2}}\s+({_MONTHS})\s+\d{{4}}\b",     # 27 January 2026
        rf"\b({_MONTHS})\s+\d{{1,2}},\s+\d{{4}}\b",   # January 27, 2026
        r"\b\d{4}-\d{2}-\d{2}\b",                      # 2026-01-27 (rare but safe)
    ]

    for pat in patterns:
        m = re.search(pat, top_text)
        if not m:
            continue
        candidate = m.group(0)
        iso = _to_iso(candidate)
        if iso:
            return iso

    return "unknown"


# ------------------ TITLE EXTRACTION ------------------ #

def _extract_title_fallback(html: str) -> str:
    """
    Robust title fallback using common meta tags and <title>.
    """
    soup = BeautifulSoup(html, "html.parser")

    for attrs in (
        {"property": "og:title"},
        {"name": "og:title"},
        {"name": "twitter:title"},
    ):
        tag = soup.find("meta", attrs=attrs)
        if tag and tag.get("content"):
            return tag["content"].strip()

    if soup.title and soup.title.string:
        return soup.title.string.strip()

    return ""


# ------------------ MAIN EXTRACTION ------------------ #

def extract_article_text(url: str, timeout: int = 20) -> dict:
    """
    Extracts and cleans the main article text and title from a URL.
    Returns a dict: {"url": str, "title": str, "text": str, "article_published_date": str}
    Never returns None.
    """

    def _prep(s: str) -> str:
        if not s:
            return ""
        s = s.replace("\u00a0", " ")
        s = s.replace("\r", " ")
        s = " ".join(s.split())
        return s.strip()

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0 Safari/537.36"
        )
    }

    title, text, published_date = "", "", "unknown"
    final_url = url

    try:
        resp = requests.get(url, headers=headers, timeout=timeout, allow_redirects=True)
        resp.raise_for_status()
        final_url = str(resp.url)
        html = resp.text

        # date: improved extraction (HTML first)
        published_date = extract_published_date_from_html(html)

        # if still unknown, use URL fallbacks (use final URL after redirects)
        if published_date == "unknown":
            published_date = fallback_date_from_url(final_url)
            if published_date == "unknown":
                published_date = fallback_date_from_url(url)

        # ---- Primary extractor: Trafilatura ----
        text = trafilatura.extract(
            html,
            include_comments=False,
            include_tables=False
        ) or ""

        # Title via trafilatura metadata if available
        meta = trafilatura.bare_extraction(html)
        if isinstance(meta, dict):
            title = meta.get("title") or ""
        else:
            title = getattr(meta, "title", "") or ""

        # Extra fallback if title missing
        if not title:
            title = _extract_title_fallback(html)

        # ---- Fallback: Newspaper3k if Trafilatura failed/short ----
        if (not text or len(text) < 200) and _HAS_NEWSPAPER:
            try:
                art = _NPArticle(final_url)
                art.download()
                art.parse()
                title = title or (art.title or "")
                text = art.text or text
            except Exception:
                pass

    except Exception:
        # ---- Final fallback ----
        if _HAS_NEWSPAPER:
            try:
                art = _NPArticle(final_url)
                art.download()
                art.parse()
                title = art.title or ""
                text = art.text or ""
            except Exception:
                pass

        # If date unknown, still try URL fallback
        if published_date == "unknown":
            published_date = fallback_date_from_url(final_url)
            if published_date == "unknown":
                published_date = fallback_date_from_url(url)

    title = _prep(title)
    text = _prep(text)

    return {"url": final_url, "title": title, "text": text, "article_published_date": published_date}


article = extract_article_text("https://www.bbc.co.uk/news/articles/c07m2v1z4evo")
if not article["text"]:
    print("Couldn't extract the article body. You can log/skip this URL.")
else:
    print(article["title"])
    print(article["article_published_date"])
    print(article["text"][:500])


Katie Razzall: A seismic moment that shows rift at top of BBC
2025-11-09
Katie Razzall: A seismic moment that shows rift at top of BBC - Published This is seismic. To lose both the director general and the CEO of BBC News at the same time is unprecedented. It's an extraordinary moment in the history of the BBC. It can't be overstated. On the face of it, Tim Davie's resignation makes some sense. I have wondered for some time whether he was weighing up how much longer he wanted to stay in a job that is highly pressurised. There have been occasions when I have interview


In [7]:
# stage2_mining_impacts_with_scrape_levels.py
# Scrapes full article text with your webscraper.extract_article_text(url),
# then uses GPT-5-nano to:
#   1) classify if it's ZAMBIA MINING related (True/False) + confidence
#   2) classify impacts in 3 levels:
#        - impact_level1  (Environmental | Social | Governance)
#        - impact_level2  (subcategory, e.g., "Water Resources")
#        - impact_level3  (specific sub-impact, e.g., "Acid mine drainage")
#      Supports MULTIPLE impacts by storing multiple triples in one cell separated by " || ".
#   3) extract evidence snippets from the scraped ARTICLE TEXT for each triple.
#
# Output: same CSV + extra columns:
#   - mining_related
#   - mining_related_confidence
#   - impact_level1
#   - impact_level2
#   - impact_level3
#   - impact_confidence
#   - impact_evidence
#   - scrape_status
#   - scraped_title
#   - scraped_published_date
#
# Conservative behavior:
# - If scrape fails / text too short: mining_related=False and no impacts.
# - Mining_related=True only if BOTH Zambia + mining/minerals are clearly present in the scraped text.
# - Impacts only if explicitly supported by text.

from __future__ import annotations

import os
import json
import time
import random
import csv
from pathlib import Path
from typing import Any, Dict, List, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm

# from webscraper import extract_article_text


# ------------------ CONFIG ------------------ #

IN_PATH = Path("data/interim/gdelt_event_context_daily/2025/12/01/zambia_events_collapsed_enriched_mining_filtered_remaining.csv")
OUT_PATH = IN_PATH.with_name(IN_PATH.stem + "_with_mining_impacts_levels.csv")

DEFAULT_MODEL = "gpt-5-nano-2025-08-07"

# scraping
SCRAPE_TIMEOUT_S = 20
MIN_TEXT_CHARS = 300
MAX_WORKERS = 10
SLEEP_BETWEEN_REQ = (0.05, 0.15)

# scrape cache (recommended)
SCRAPE_CACHE_PATH = IN_PATH.with_name(IN_PATH.stem + "_scrape_cache.csv")
SCRAPE_CACHE_FIELDS = [
    "url_normalized",
    "final_url",
    "scrape_ok",
    "scrape_status",
    "scraped_title",
    "scraped_published_date",
    "text",
]

# prompt sizing
MAX_TEXT_CHARS_FOR_LLM = 14000  # keep bounded for cost


# ------------------ OPENAI SETUP ------------------ #

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY not found in environment or .env file.")
client = OpenAI(api_key=api_key)


# ------------------ IMPACT TAXONOMY (3 levels) ------------------ #

IMPACT_TAXONOMY: Dict[str, Dict[str, List[str]]] = {
    "Environmental": {
        "Land, Soil & Geology": [
            "Overburden removal and waste rock generation",
            "Tailings generation and management",
            "Topsoil degradation and loss of soil fertility",
            "Deep soil compaction and subsurface degradation",
            "Land disturbance and land-use change",
            "Landscape and topography alteration",
            "Surface ground instability and subsidence",
            "Slope instability and landslides",
            "Fly rock from blasting",
            "Ground vibration and air overpressure (blasting impacts)",
            "Transport corridors and access road impacts",
            "Geothermal effects at depth (elevated subsurface temperatures)",
        ],
        "Water Resources": [
            "Acid mine drainage",
            "Surface water pollution and quality degradation",
            "Groundwater contamination",
            "Freshwater ecotoxicity (heavy metals, pH changes, sensitive species)",
            "Eutrophication of water bodies",
            "Water table alteration and drainage issues",
            "High water consumption",
            "Water management challenges",
        ],
        "Air & Climate": [
            "Air quality degradation (dust, diesel exhaust, blasting emissions)",
            "Particulate matter emissions",
            "Dust generation",
            "Photochemical ozone formation (smog)",
            "Ozone depletion (emissions from equipment/explosives)",
            "Greenhouse gas emissions (GHGs)",
            "Local atmospheric heating and microclimate change",
            "Temperature inversions trapping pollution",
            "Fossil fuel consumption",
            "Energy demand and consumption",
            "Clean/renewable energy integration (mitigation-related impact)",
            "Carbon sink destruction (deforestation, land clearing)",
        ],
        "Biodiversity & Ecosystems": [
            "Ecosystem degradation and habitat loss/fragmentation",
            "Deforestation",
            "Impacts on terrestrial ecotoxicity",
            "Impacts on aquatic ecosystems and life below water",
            "Loss of ecosystem services",
            "Ecosystem recovery through reclamation and restoration (post-mining impact)",
        ],
        "Noise & Disturbance": [
            "Noise pollution (drilling, blasting, crushing, hauling)",
        ],
    },
    "Social": {
        "Health, Safety & Well-being": [
            "Worker health and safety risks",
            "Community exposure to dust, noise, pollution",
            "Accidents and fatalities (including structural failures such as tailings dams)",
            "Equipment safety risks",
            "Material safety risks",
            "Mental and physical health impacts on communities",
            "Public safety concerns from blasting, vibrations, fly rock",
        ],
        "Livelihoods & Economy": [
            "Employment creation and job security",
            "Unemployment risks post-closure",
            "Income generation and poverty reduction or exacerbation",
            "Business opportunities (local supply chains, services)",
            "Human capital development (skills, training, workforce development)",
            "Equipment and materials availability affecting local labor markets",
        ],
        "Living Conditions & Social Fabric": [
            "Livability (cost of living, food security, communication services)",
            "Social relationships and family/community cohesion",
            "Demographic changes (migration, influx of workers, displacement)",
            "Social infrastructure and amenities (education, healthcare, public services, leisure)",
            "Education and training opportunities",
            "Tourism attraction or decline",
            "Cultural impacts (tangible and intangible heritage, traditions, identity)",
        ],
        "Equity, Rights & Vulnerable Groups": [
            "Freedom and justice impacts in nearby communities",
            "Stakeholder inclusion and community participation",
            "Future generations’ rights and intergenerational equity",
            "Land ownership impacts and regional value changes",
            "Indigenous peoples’ rights and conflicts with mining",
            "Child labor and forced labor risks",
            "Crime and social disorder (corruption, violence, substance abuse)",
            "Wealth distribution and inequality",
        ],
        "Quality of Work Life": [
            "Job satisfaction",
            "Labor conditions and workplace dignity",
        ],
    },
    "Governance": {
        "Regulation, Compliance & Institutions": [
            "Legislative and regulatory framework effectiveness",
            "Enforcement of environmental and social regulations",
            "Mining permitting and licensing practices",
            "Compliance with national and international standards",
        ],
        "Transparency & Accountability": [
            "Corporate transparency and reporting",
            "Monitoring and disclosure of environmental and social performance",
            "Anti-corruption and bribery risks",
            "Accountability for accidents and environmental damage",
        ],
        "Stakeholder Engagement & Rights": [
            "Stakeholder inclusion in decision-making",
            "Community consent and consultation (FPIC – Free, Prior and Informed Consent)",
            "Conflict management between companies and communities",
            "Protection of indigenous land and resource rights",
        ],
        "Risk Management & Long-Term Stewardship": [
            "Tailings and waste governance",
            "Closure planning and post-mining reclamation governance",
            "Long-term liability management (pollution, water treatment, land stability)",
            "Intergenerational responsibility and resource depletion ethics",
        ],
        "Reputation & Social License": [
            "Mining image and public perception",
            "Social license to operate",
            "Trust between mining companies, government, and communities",
        ],
    },
}

# Flatten allowed triples for validation
ALLOWED_TRIPLES = set()
for l1, sub in IMPACT_TAXONOMY.items():
    for l2, l3s in sub.items():
        for l3 in l3s:
            ALLOWED_TRIPLES.add((l1, l2, l3))


# ------------------ HELPERS ------------------ #

def _norm_str(v: Any) -> str:
    if v is None:
        return ""
    return str(v).strip()

def _truncate(s: str, n: int) -> str:
    s = s or ""
    return s[:n] if len(s) > n else s

def _to_bool(v: Any) -> bool:
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return v != 0
    if isinstance(v, str):
        return v.strip().lower() in {"true", "1", "yes"}
    return False

def _safe_float(v: Any, default: float = 0.0) -> float:
    try:
        return float(v)
    except Exception:
        return default

def _build_llm_text(url: str, scraped_title: str, published_date: str, text: str) -> str:
    text = _truncate(text, MAX_TEXT_CHARS_FOR_LLM)
    return f"URL: {url}\nPUBLISHED_DATE: {published_date}\nTITLE: {scraped_title}\n\nTEXT:\n{text}".strip()


# ------------------ SCRAPE CACHE ------------------ #

def load_scrape_cache() -> Dict[str, Dict[str, str]]:
    cache: Dict[str, Dict[str, str]] = {}
    if SCRAPE_CACHE_PATH.exists():
        with open(SCRAPE_CACHE_PATH, "r", encoding="utf-8") as f:
            for r in csv.DictReader(f):
                u = (r.get("url_normalized") or "").strip()
                if u:
                    cache[u] = r
    return cache

def append_scrape_cache_row(row: Dict[str, Any]) -> None:
    write_header = not SCRAPE_CACHE_PATH.exists()
    with open(SCRAPE_CACHE_PATH, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=SCRAPE_CACHE_FIELDS)
        if write_header:
            w.writeheader()
        out = {k: row.get(k, "") for k in SCRAPE_CACHE_FIELDS}
        w.writerow(out)


# ------------------ LLM CLASSIFIER ------------------ #

def llm_classify_mining_and_impacts_3level(
    url: str,
    scraped_title: str,
    published_date: str,
    article_text: str,
    model: str = DEFAULT_MODEL,
    timeout: int = 75,
) -> Dict[str, Any]:
    """
    Returns:
      {
        "mining_related": bool,
        "mining_related_confidence": float,
        "impacts": [{"level1":..., "level2":..., "level3":...}, ...],
        "impact_confidence": float,
        "impact_evidence": [{"level1":..,"level2":..,"level3":..,"snippets":[...]} ...]
      }
    """
    if not article_text or len(article_text) < MIN_TEXT_CHARS:
        return {
            "mining_related": False,
            "mining_related_confidence": 0.0,
            "impacts": [],
            "impact_confidence": 0.0,
            "impact_evidence": [],
        }

    system_prompt = (
        "You are a conservative classification and evidence extraction engine.\n"
        "Return ONE JSON object and NOTHING else."
    )

    user_prompt = f"""
We are extracting labels from news articles.

Task A (binary, conservative):
Decide if the article relates to ZAMBIA MINING (true/false).
Return TRUE only if BOTH are clearly supported by the text:
  (1) Zambia context is clear (Zambia, Zambian locations, Zambian institutions, Zambian mines, etc.)
  (2) Mining/minerals/metals extraction, exploration, processing, licensing, transport/export of ore/concentrates,
      mine accidents, labour disputes at mines, or mining regulation are clearly involved.
If unsure about either (1) or (2) => FALSE.

Task B (multi-label impacts, 3-level):
If mining_related is TRUE, identify which mining impacts (0, 1, or many) are explicitly mentioned.
Each impact must be output as three levels:
  - level1: Environmental | Social | Governance
  - level2: a subcategory within that level1 (must match allowed taxonomy)
  - level3: the specific impact item (must match allowed taxonomy)
Only include impacts explicitly supported by the text. Do NOT infer.

Evidence:
For each selected impact, provide 1–3 short verbatim snippets (<= 25 words each) from the TEXT that justify it.

Output JSON with exactly:
{{
  "mining_related": true|false,
  "mining_related_confidence": 0.0,
  "impacts": [
    {{"level1":"Environmental|Social|Governance", "level2":"...", "level3":"..."}}
  ],
  "impact_confidence": 0.0,
  "impact_evidence": [
    {{
      "level1":"...", "level2":"...", "level3":"...",
      "snippets": ["<verbatim snippet>", "..."]
    }}
  ]
}}

Rules:
- Use ONLY the article text provided.
- If mining_related is FALSE: impacts=[], impact_evidence=[], impact_confidence=0.0.
- level1/level2/level3 MUST come from the allowed taxonomy below (exact strings).
- mining_related_confidence: confidence in the mining_related decision (0–1).
- impact_confidence: confidence the impacts list is correct (0–1).

Allowed taxonomy (exact strings):
{json.dumps(IMPACT_TAXONOMY, ensure_ascii=False)}

{_build_llm_text(url, scraped_title, published_date, article_text)}
""".strip()

    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format={"type": "json_object"},
        timeout=timeout,
    )

    raw = completion.choices[0].message.content or "{}"
    data = json.loads(raw) if raw else {}

    mining_related = _to_bool(data.get("mining_related", False))
    mining_conf = max(0.0, min(1.0, _safe_float(data.get("mining_related_confidence", 0.0), 0.0)))

    if not mining_related:
        return {
            "mining_related": False,
            "mining_related_confidence": mining_conf,
            "impacts": [],
            "impact_confidence": 0.0,
            "impact_evidence": [],
        }

    impacts = data.get("impacts", [])
    if not isinstance(impacts, list):
        impacts = []

    impacts_clean: List[Dict[str, str]] = []
    seen = set()
    for item in impacts:
        if not isinstance(item, dict):
            continue
        l1 = _norm_str(item.get("level1"))
        l2 = _norm_str(item.get("level2"))
        l3 = _norm_str(item.get("level3"))
        if (l1, l2, l3) in ALLOWED_TRIPLES:
            key = (l1, l2, l3)
            if key not in seen:
                seen.add(key)
                impacts_clean.append({"level1": l1, "level2": l2, "level3": l3})

    impact_conf = max(0.0, min(1.0, _safe_float(data.get("impact_confidence", 0.0), 0.0)))

    evidence = data.get("impact_evidence", [])
    if not isinstance(evidence, list):
        evidence = []

    ev_clean: List[Dict[str, Any]] = []
    for item in evidence:
        if not isinstance(item, dict):
            continue
        l1 = _norm_str(item.get("level1"))
        l2 = _norm_str(item.get("level2"))
        l3 = _norm_str(item.get("level3"))
        if (l1, l2, l3) not in ALLOWED_TRIPLES:
            continue
        snips = item.get("snippets", [])
        if not isinstance(snips, list):
            snips = []
        snips_clean = []
        for s in snips:
            ss = _norm_str(s)
            if ss:
                snips_clean.append(_truncate(ss, 250))
        if snips_clean:
            ev_clean.append({"level1": l1, "level2": l2, "level3": l3, "snippets": snips_clean})

    return {
        "mining_related": True,
        "mining_related_confidence": mining_conf,
        "impacts": impacts_clean,
        "impact_confidence": impact_conf,
        "impact_evidence": ev_clean,
    }


# ------------------ SCRAPE + PIPELINE ------------------ #

def scrape_one(url_norm: str, url: str, scrape_cache: Dict[str, Dict[str, str]]) -> Dict[str, Any]:
    # cache hit (only if scrape_ok and enough text)
    if url_norm in scrape_cache:
        c = scrape_cache[url_norm]
        ok = str(c.get("scrape_ok", "")).strip().lower() in {"true", "1", "yes"}
        text = c.get("text", "") or ""
        if ok and len(text) >= MIN_TEXT_CHARS:
            return {
                "final_url": c.get("final_url", url),
                "scrape_ok": True,
                "scrape_status": c.get("scrape_status", "ok_cache"),
                "scraped_title": c.get("scraped_title", ""),
                "scraped_published_date": c.get("scraped_published_date", "unknown"),
                "text": text,
            }

    time.sleep(random.uniform(*SLEEP_BETWEEN_REQ))

    try:
        art = extract_article_text(url, timeout=SCRAPE_TIMEOUT_S)
        text = _norm_str(art.get("text"))
        title = _norm_str(art.get("title"))
        pub = _norm_str(art.get("article_published_date")) or "unknown"
        final_url = _norm_str(art.get("url")) or url

        ok = len(text) >= MIN_TEXT_CHARS
        status = "ok" if ok else "empty_or_short_text"

        out = {
            "final_url": final_url,
            "scrape_ok": ok,
            "scrape_status": status,
            "scraped_title": title,
            "scraped_published_date": pub,
            "text": text,
        }

        if ok:
            append_scrape_cache_row({
                "url_normalized": url_norm,
                "final_url": final_url,
                "scrape_ok": True,
                "scrape_status": status,
                "scraped_title": title,
                "scraped_published_date": pub,
                "text": text,
            })

        return out

    except Exception as e:
        return {
            "final_url": url,
            "scrape_ok": False,
            "scrape_status": f"scrape_exception:{type(e).__name__}",
            "scraped_title": "",
            "scraped_published_date": "unknown",
            "text": "",
        }


def run_stage2(
    in_path: Path = IN_PATH,
    out_path: Path = OUT_PATH,
    model: str = DEFAULT_MODEL,
    max_rows: Optional[int] = None,
) -> None:
    if not in_path.exists():
        raise FileNotFoundError(f"Input not found: {in_path}")

    df = pd.read_csv(in_path)
    if max_rows is not None:
        df = df.head(max_rows).copy()

    if "sourceurl" not in df.columns:
        raise ValueError("Input CSV must include a 'sourceurl' column.")
    if "url_normalized" not in df.columns:
        # fallback if missing
        df["url_normalized"] = df["sourceurl"].astype(str)

    # Add output columns (3-level)
    df["mining_related"] = False
    df["mining_related_confidence"] = 0.0
    df["impact_level1"] = ""
    df["impact_level2"] = ""
    df["impact_level3"] = ""
    df["impact_confidence"] = 0.0
    df["impact_evidence"] = ""

    # scrape metadata columns (handy for debugging/audit)
    df["scrape_status"] = ""
    df["scraped_title"] = ""
    df["scraped_published_date"] = ""

    rows = df.to_dict(orient="records")
    scrape_cache = load_scrape_cache()

    print(f"Stage2 (scrape + LLM, 3-level): {len(rows):,} rows | model={model} | workers={MAX_WORKERS}")

    # 1) Scrape in parallel
    scraped_results: Dict[int, Dict[str, Any]] = {}

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {}
        for idx, r in enumerate(rows):
            url = _norm_str(r.get("sourceurl"))
            url_norm = _norm_str(r.get("url_normalized")) or url
            futures[ex.submit(scrape_one, url_norm, url, scrape_cache)] = idx

        for fut in tqdm(as_completed(futures), total=len(futures), desc="Scraping"):
            idx = futures[fut]
            scraped_results[idx] = fut.result()

    # 2) LLM classify
    for idx, r in enumerate(tqdm(rows, desc="LLM classify")):
        s = scraped_results.get(idx, {})
        r["scrape_status"] = s.get("scrape_status", "no_scrape_result")
        r["scraped_title"] = s.get("scraped_title", "")
        r["scraped_published_date"] = s.get("scraped_published_date", "unknown")

        if not s.get("scrape_ok", False):
            # Conservative: no scrape => don't label mining/impacts
            continue

        url = _norm_str(r.get("sourceurl"))
        result = llm_classify_mining_and_impacts_3level(
            url=url,
            scraped_title=r["scraped_title"],
            published_date=r["scraped_published_date"],
            article_text=s.get("text", ""),
            model=model,
        )

        r["mining_related"] = bool(result["mining_related"])
        r["mining_related_confidence"] = round(float(result["mining_related_confidence"]), 3)
        r["impact_confidence"] = round(float(result.get("impact_confidence", 0.0)), 3)

        # Flatten MULTIPLE impacts into 3 columns by joining with " || "
        impacts = result.get("impacts", [])
        if impacts:
            r["impact_level1"] = " || ".join([i["level1"] for i in impacts])
            r["impact_level2"] = " || ".join([i["level2"] for i in impacts])
            r["impact_level3"] = " || ".join([i["level3"] for i in impacts])
        else:
            r["impact_level1"] = ""
            r["impact_level2"] = ""
            r["impact_level3"] = ""

        # Flatten evidence: [L1 > L2 > L3] “snippet” | “snippet” || ...
        ev_items = result.get("impact_evidence", [])
        flat_parts = []
        if isinstance(ev_items, list):
            for item in ev_items:
                l1 = _norm_str(item.get("level1"))
                l2 = _norm_str(item.get("level2"))
                l3 = _norm_str(item.get("level3"))
                snips = item.get("snippets", [])
                if l1 and l2 and l3 and isinstance(snips, list) and snips:
                    joined = " | ".join([f"“{_norm_str(s)}”" for s in snips if _norm_str(s)])
                    if joined:
                        flat_parts.append(f"[{l1} > {l2} > {l3}] {joined}")
        r["impact_evidence"] = " || ".join(flat_parts)

    out_df = pd.DataFrame(rows)
    out_df.to_csv(out_path, index=False, encoding="utf-8")

    print(f"\nSaved: {out_path}")
    print("Mining-related True:", int(out_df["mining_related"].sum()))
    any_impact = (out_df["impact_level3"].fillna("").astype(str).str.strip() != "")
    print("With ≥1 impact:", int(any_impact.sum()))
    print(f"Scrape cache: {SCRAPE_CACHE_PATH}")


In [8]:
run_stage2()

Stage2 (scrape + LLM, 3-level): 10 rows | model=gpt-5-nano-2025-08-07 | workers=10


LLM classify: 100%|██████████| 10/10 [04:49<00:00, 28.98s/it]


Saved: data/interim/gdelt_event_context_daily/2025/12/01/zambia_events_collapsed_enriched_mining_filtered_remaining_with_mining_impacts_levels.csv
Mining-related True: 5
With ≥1 impact: 4
Scrape cache: data/interim/gdelt_event_context_daily/2025/12/01/zambia_events_collapsed_enriched_mining_filtered_remaining_scrape_cache.csv


These 10 articles cost a $0.01 or less